# HCMC flood observations — Method 2 / Random Forest pilot

Adaptation of the supplied **Phase 3 RF notebook** (`notebook50b59dca3c.ipynb`). Retains RandomForestRegressor with 10 trees and seed 42; replaces raster sampling and 13 Berlin predictors with a reader and explicit feature selection for the original HCMC tabular file. Trains a new RF; Berlin weights are not transferred. The original data file remains unchanged.

**Target:** the reported upper depth endpoint, `Water height max (cm)` / 100. This is NOT necessarily measured peak flood depth. Missing upper endpoints are excluded, never filled using the lower endpoint or a class. This pilot describes severity among reported flood observations, not flood occurrence or a physical rainfall forecast.

Source: Hardy & Radziute (2026), https://doi.org/10.23708/8Y16HU, V1, CC BY-NC 4.0. Dataset contains historical newspaper/report observations. Code reference: https://github.com/omarseleem92/Machine_learning_for_urban_flooding ; paper https://doi.org/10.5194/nhess-23-809-2023 .

Kaggle: CPU is sufficient. Put `Flood_event_locations.tab` beside this notebook or attach it as an input. If absent, the notebook downloads the public original file (Internet required). Run all cells. Dependencies: numpy, pandas, scikit-learn.


In [1]:
from pathlib import Path
import json, hashlib, urllib.request, unicodedata, re, platform
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT = BASE / 'hcm_method2_results'
OUT.mkdir(exist_ok=True)
DATA = BASE / 'Flood_event_locations.tab'
if not DATA.exists() and Path('/kaggle/input').exists():
    candidates = sorted(Path('/kaggle/input').rglob('Flood_event_locations.tab'))
    if candidates: DATA = candidates[0]
if not DATA.exists():
    data_bytes = urllib.request.urlopen('https://dataverse.ird.fr/api/access/datafile/50148', timeout=60).read()
    DATA.write_bytes(data_bytes)
SOURCE_HASH = hashlib.sha256(DATA.read_bytes()).hexdigest()
raw = pd.read_csv(DATA, sep='\t', dtype=str, keep_default_na=False)
print('Original dataset:', raw.shape, '| SHA256:', SOURCE_HASH)
print('Versions:', platform.python_version(), pd.__version__, sklearn.__version__)


Original dataset: (425, 22) | SHA256: 00c96766d0cc8f794ae518929eaf58763d1a36354b678dad6ea00cb7a9d68962
Versions: 3.11.0 3.1.0 1.9.1


## Audit and target
Rows without a usable upper endpoint are excluded. A repeated Obs_id is reported but is not automatically considered a duplicated record. Exact duplicate substantive rows are removed only in memory. No data-dependent model selection uses the evaluation folds.


In [2]:
target_col = 'Water height max (cm)'
upper = pd.to_numeric(raw[target_col], errors='coerce')
lower = pd.to_numeric(raw['Water height min (cm)'], errors='coerce')
valid = upper.notna() & (upper >= 0) & (~lower.notna() | (lower <= upper))
df = raw.loc[valid].copy()
substantive = [c for c in raw.columns if c not in ['OBJECTID', 'Obs_id']]
n_before = len(df)
df = df.drop_duplicates(subset=substantive).reset_index(drop=True)
y = pd.to_numeric(df[target_col]).to_numpy(dtype=float) / 100.0
audit = {'original_rows':len(raw), 'usable_upper_endpoint_rows':int(valid.sum()),
         'excluded_rows':int((~valid).sum()), 'exact_duplicate_rows_removed':n_before-len(df),
         'duplicate_obs_id_rows':int(raw['Obs_id'].duplicated().sum()),
         'target_min_m':float(y.min()), 'target_max_m':float(y.max()),
         'cause_counts':df['Cause'].value_counts().to_dict(), 'sha256':SOURCE_HASH}
print(json.dumps(audit, ensure_ascii=False, indent=2))
(OUT/'data_audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding='utf-8')


{
  "original_rows": 425,
  "usable_upper_endpoint_rows": 147,
  "excluded_rows": 278,
  "exact_duplicate_rows_removed": 0,
  "duplicate_obs_id_rows": 1,
  "target_min_m": 0.1,
  "target_max_m": 3.0,
  "cause_counts": {
    "High tide": 76,
    "Rain": 54,
    "Combined": 17
  },
  "sha256": "00c96766d0cc8f794ae518929eaf58763d1a36354b678dad6ea00cb7a9d68962"
}


## Features and grouped evaluation
Main feature set: year, month (0 if unknown), location type, district and province. An additional retrospective variant includes the reported cause (rain/tide/combined); it is usable only when that cause is known. No water-depth columns, depth class, impacts, notes, duration, identifiers or source fields are input features.

Primary evaluation groups together observations sharing a source OR a known date, transitively. This reduces reuse of the same report/event across training and testing. Dates missing in the original remain unknown; the grouping is a conservative proxy, not verified event IDs. A separate location-group evaluation tests unseen named locations, but may share events/sources. These protocols answer different questions. No claim of simultaneous spatial and event independence is made.


In [3]:
def norm(v):
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFC',str(v)).strip().lower()) or '__missing__'

dates = pd.to_datetime(df['Date'], format='%d/%m/%Y', errors='coerce')
X = pd.DataFrame({'Year':pd.to_numeric(df['Year'],errors='coerce').fillna(0),
                  'Month':dates.dt.month.fillna(0)})
cats = ['Location type','District mentioned','Province']
for c in cats+['Cause']: X[c] = df[c].map(norm)

parent = list(range(len(df)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
seen = {}
for i in range(len(df)):
    keys=[]
    if df.loc[i,'Source id'].strip(): keys.append('source:'+norm(df.loc[i,'Source id']))
    if pd.notna(dates.iloc[i]): keys.append('date:'+dates.iloc[i].strftime('%Y-%m-%d'))
    for key in keys:
        if key in seen: parent[find(i)] = find(seen[key])
        else: seen[key]=i
event_groups=np.array([find(i) for i in range(len(df))])
location_groups=df['Location name'].map(norm).to_numpy()
print('Event/source components:',len(set(event_groups)))
print('Named locations:',len(set(location_groups)))
print('Unparsed or missing dates:',int(dates.isna().sum()))


Event/source components: 23
Named locations: 97
Unparsed or missing dates: 24


In [4]:
scores=[]
predictions=[]
fold_scores=[]
for protocol,groups in [('event_source',event_groups),('location',location_groups)]:
    n_splits=min(5,len(set(groups)))
    if n_splits < 3: raise ValueError('Too few independent groups')
    splits=list(GroupKFold(n_splits=n_splits).split(X,y,groups))
    for model_name in ['mean_baseline','rf_context','rf_context_and_known_cause']:
        categorical=cats+(['Cause'] if model_name.endswith('known_cause') else [])
        columns=['Year','Month']+categorical
        oof=np.full(len(y),np.nan)
        for fold,(train,test) in enumerate(splits):
            assert not set(groups[train]) & set(groups[test])
            if model_name=='mean_baseline':
                model=DummyRegressor(strategy='mean')
            else:
                prep=ColumnTransformer([('cat',OneHotEncoder(handle_unknown='ignore'),categorical),
                                        ('num','passthrough',['Year','Month'])])
                model=Pipeline([('prepare',prep),('rf',RandomForestRegressor(
                    n_estimators=10,random_state=42,n_jobs=4))])
            model.fit(X.iloc[train][columns], y[train])
            pred=model.predict(X.iloc[test][columns])
            assert np.isfinite(pred).all()
            oof[test]=pred
            fold_scores.append(dict(protocol=protocol,model=model_name,fold=fold,n_train=len(train),
                                    n_test=len(test),mae_m=mean_absolute_error(y[test],pred),
                                    rmse_m=float(np.sqrt(mean_squared_error(y[test],pred)))))
            for i,p in zip(test,pred):
                predictions.append(dict(protocol=protocol,model=model_name,fold=fold,
                    object_id=df.loc[i,'OBJECTID'],obs_id=df.loc[i,'Obs_id'],
                    actual_upper_m=y[i],predicted_upper_m=float(p)))
        assert np.isfinite(oof).all()
        scores.append(dict(protocol=protocol,model=model_name,n=len(y),n_folds=n_splits,
            mae_m=mean_absolute_error(y,oof),rmse_m=float(np.sqrt(mean_squared_error(y,oof))),
            r2=r2_score(y,oof)))
metrics=pd.DataFrame(scores)
print(metrics.to_string(index=False))
metrics.to_csv(OUT/'metrics.csv',index=False)
pd.DataFrame(fold_scores).to_csv(OUT/'fold_metrics.csv',index=False)
pd.DataFrame(predictions).to_csv(OUT/'out_of_fold_predictions.csv',index=False)
assert hashlib.sha256(DATA.read_bytes()).hexdigest()==SOURCE_HASH


    protocol                      model   n  n_folds    mae_m   rmse_m        r2
event_source              mean_baseline 147        5 0.261063 0.494804 -0.110866
event_source                 rf_context 147        5 0.289284 0.497594 -0.123431
event_source rf_context_and_known_cause 147        5 0.304824 0.510204 -0.181089
    location              mean_baseline 147        5 0.234148 0.471400 -0.008265
    location                 rf_context 147        5 0.096964 0.254589  0.705913
    location rf_context_and_known_cause 147        5 0.097837 0.258086  0.697779


## Interpretation
Compare each RF with `mean_baseline` on the **same protocol**. Lower MAE/RMSE is better. R² < 0 means errors exceed predicting the overall evaluation-target mean; the separately fitted training-mean baseline is the operational comparator. Few labels, heterogeneous sources, coarse administrative features, reporting bias, and uncertain depth endpoints limit interpretation. Fold scores expose variability; this is a pilot, not a validated flood forecast.

The target and inputs differ from Berlin, so Berlin/HCMC metric values are not a direct transfer-performance comparison. U-Net would additionally require spatial input layers and a loss appropriate for sparse point labels. Keep the Berlin U-Net result as a separate experiment.


In [5]:
summary={'status':'EXECUTED_HCMC_RF_METHOD2_PILOT',
 'source_notebook':'notebook50b59dca3c.ipynb (Berlin Phase 3 RF)',
 'adaptation':'new RF trained on HCMC tabular observations; no pretrained Berlin weights',
 'target':'reported upper depth endpoint in metres; not necessarily observed peak',
 'features_main':['Year','Month']+cats, 'extra_retrospective_feature':'Cause',
 'audit':audit, 'metrics':metrics.to_dict(orient='records'),
 'versions':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,'sklearn':sklearn.__version__}}
(OUT/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print('Finished. Outputs:',OUT)


Finished. Outputs: C:\Users\ADMIN\.codex\.chatgpt-projects\g-p-6ace0b9440e881919ba173b2df63a9ac\hcm_method2\hcm_method2_results


## Observed result from this run

Primary event/source grouping: context RF MAE 0.2893 m, RMSE 0.4976 m, R² -0.1234; mean baseline MAE 0.2611 m, RMSE 0.4948 m. RF did not improve on the mean baseline. Adding known Cause also failed to improve.

Location grouping gives context RF R² 0.7059, but may share reporting events/sources across folds. This result does not override the failed event/source generalization check. Method 2 executes successfully; predictive usefulness for independent flood events is not established.

Executed locally in Python, not on Kaggle, using 147 labeled observations across 23 event/source components.